# ComfyUI Telegram Бот — Colab T4

Запустите **Telegram бота**, который генерирует видео и фото с помощью ComfyUI на бесплатном GPU T4.

```
╔══════════╗    ╔══════════════╗    ╔══════════╗    ╔════════════╗    ╔═══════════════╗
║  1. GPU  ║ -> ║ 2. Установка ║ -> ║ 3.Модели ║ -> ║ 4.Воркфлоу ║ -> ║ 5. ЗАПУСК БОТА║ -> Telegram!
╚══════════╝    ╚══════════════╝    ╚══════════╝    ╚════════════╝    ╚═══════════════╝
    ~5 сек         ~2 мин             ~10 мин          ~30 сек            ~1 мин
```

---

### Что нужно до запуска?

> **Получите токен бота у [@BotFather](https://t.me/BotFather) в Telegram — это займёт 1 минуту:**
>
> 1. Откройте [@BotFather](https://t.me/BotFather) в Telegram
> 2. Отправьте `/newbot`
> 3. Придумайте **имя** бота (например: `Мой Видео Бот`)
> 4. Придумайте **username** (например: `my_comfy_video_bot`)
> 5. Скопируйте **токен** вида `1234567890:ABCdefGhIjKlMnOpQrStUvWxYz`
>
> Токен понадобится в ячейке 5.

---

### Команды бота

| Команда | Что отправить | Воркфлоу | Результат |
|:--------|:--------------|:---------|:----------|
| `/photo <промпт>` | Текст + ответ на фото | `wan_clip` | Короткое видео 3.4с из изображения |
| `/text <промпт>` | Только текст | `wan_t2v` | Видео из текстового описания |
| `/talking` | Ответ на фото + аудио | `wan_talking` | Говорящая голова |
| `/v2v <промпт>` | Текст + ответ на видео | `wan_v2v` | Рестайл видео |
| `/status` | — | — | Проверить статус очереди |
| `/cancel` | — | — | Отменить текущую задачу |

### Ограничения

| Параметр | Значение |
|:---------|:---------|
| Время сессии | Макс. 12ч, отключение через 90мин простоя |
| Генерация | 2-10 мин на видео (T4 16ГБ) |
| Параллельность | Одна задача за раз (ограничение VRAM) |
| Размер видео | Telegram лимит — 50 МБ |

---

**Запускайте ячейки 1-5 по порядку.**

In [ ]:
#@title 1. Проверка GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} ГБ")

In [ ]:
#@title 2. Установка ComfyUI + Кастомные ноды + Зависимости бота
%cd /content
!test -d ComfyUI || git clone https://github.com/comfyanonymous/ComfyUI.git
%cd /content/ComfyUI

# Фиксируем numpy < 2.0
!echo "numpy<2.0" > /tmp/numpy_constraint.txt
!pip install "numpy>=1.26,<2.0" -q
!pip install -r requirements.txt -q -c /tmp/numpy_constraint.txt

# Кастомные ноды (те же, что в wan ноутбуке)
%cd /content/ComfyUI/custom_nodes
!test -d ComfyUI-WanVideoWrapper || git clone https://github.com/kijai/ComfyUI-WanVideoWrapper.git
!test -d ComfyUI-VideoHelperSuite || git clone https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git
!test -d ComfyUI-KJNodes || git clone https://github.com/kijai/ComfyUI-KJNodes.git

!pip install -r ComfyUI-WanVideoWrapper/requirements.txt -q -c /tmp/numpy_constraint.txt
!pip install -r ComfyUI-VideoHelperSuite/requirements.txt -q -c /tmp/numpy_constraint.txt
!pip install -r ComfyUI-KJNodes/requirements.txt -q -c /tmp/numpy_constraint.txt

# Зависимости бота
!pip install python-telegram-bot==21.6 requests -q

print("\nВсё установлено!")

In [ ]:
#@title 3. Скачивание моделей (~28 ГБ)
import os

M = "/content/ComfyUI/models"
os.makedirs(f"{M}/diffusion_models", exist_ok=True)
os.makedirs(f"{M}/text_encoders", exist_ok=True)
os.makedirs(f"{M}/vae", exist_ok=True)

HF = "https://huggingface.co/Kijai/WanVideo_comfy/resolve/main"

files = {
    f"{M}/diffusion_models/Wan2_2-T2V-A14B-LOW_fp8_e4m3fn_scaled_KJ.safetensors":
        f"{HF}/Wan2_2-T2V-A14B-LOW_fp8_e4m3fn_scaled_KJ.safetensors",
    f"{M}/diffusion_models/Wan2_2_Fun_VACE_module_A14B_LOW_bf16.safetensors":
        f"{HF}/Wan2_2_Fun_VACE_module_A14B_LOW_bf16.safetensors",
    f"{M}/diffusion_models/fantasytalking_fp16.safetensors":
        f"{HF}/fantasytalking_fp16.safetensors",
    f"{M}/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors":
        f"{HF}/umt5_xxl_fp8_e4m3fn_scaled.safetensors",
    f"{M}/vae/wan2.2_vae.safetensors":
        f"{HF}/wan2.2_vae.safetensors",
}

for dest, url in files.items():
    if os.path.exists(dest):
        print(f"Уже существует: {os.path.basename(dest)}")
    else:
        print(f"\nСкачивание {os.path.basename(dest)}...")
        !wget -q --show-progress -O "{dest}" "{url}"

# Валидация скачивания
for dest in files:
    if os.path.exists(dest) and os.path.getsize(dest) < 1024:
        os.remove(dest)
        print(f"  ОШИБКА: {os.path.basename(dest)} не скачан — перезапустите ячейку")

print("\nВсе модели готовы!")

In [ ]:
#@title 4. Скачивание воркфлоу
import os

GIST = "ea1ab4a53e1be1b8401e5031bcdae4f3"
RAW = f"https://gist.githubusercontent.com/russiankendricklamar/{GIST}/raw"
WF_DIR = "/content/ComfyUI/user/default/workflows"
os.makedirs(WF_DIR, exist_ok=True)

workflows = [
    "video_wan_t2v.json",
    "video_wan_clip.json",
    "video_wan_v2v.json",
    "video_wan_talking.json",
]

for wf in workflows:
    !wget -q -O "{WF_DIR}/{wf}" "{RAW}/{wf}"
    print(f"Скачано: {wf}")

print(f"\n{len(workflows)} воркфлоу готовы")

In [ ]:
#@title 5. Настройка и запуск бота
#@markdown ---
#@markdown ### Как получить токен бота (1 минута)
#@markdown
#@markdown 1. Откройте [@BotFather](https://t.me/BotFather) в Telegram
#@markdown 2. Отправьте `/newbot`
#@markdown 3. Придумайте имя и username для бота
#@markdown 4. Скопируйте токен вида `1234567890:ABCdefGhIjKlMnOpQrStUvWxYz`
#@markdown 5. Вставьте его в поле ниже
#@markdown
#@markdown ---
#@markdown ### Как работает бот
#@markdown ```
#@markdown Telegram        Bot           ComfyUI         Telegram
#@markdown   |              |               |               |
#@markdown   |-- команда -->|               |               |
#@markdown   |              |-- воркфлоу -->|               |
#@markdown   |              |               |-- генерация   |
#@markdown   |              |               |   (2-10 мин)  |
#@markdown   |              |<-- видео -----|               |
#@markdown   |              |--------------- видео -------->|
#@markdown ```
#@markdown
#@markdown ---
#@markdown ### Что произойдёт после запуска
#@markdown - Запустится **ComfyUI** на порту 8188 (~30 сек)
#@markdown - Откроется **Cloudflare туннель** для веб-интерфейса
#@markdown - Запустится **Telegram бот** в режиме polling
#@markdown - В выводе появится ссылка на интерфейс ComfyUI
#@markdown - Отправьте `/start` боту в Telegram для начала!

import getpass
BOT_TOKEN = getpass.getpass("Вставьте токен Telegram бота: ")

import subprocess, time, os, json, glob, asyncio, re, shutil, requests
from telegram import Update, Bot
from telegram.ext import Application, CommandHandler, MessageHandler, filters, ContextTypes

# ── ComfyUI API Client ──────────────────────────────────────────────

COMFY_URL = "http://127.0.0.1:8188"
INPUT_DIR = "/content/ComfyUI/input"
OUTPUT_DIR = "/content/ComfyUI/output"
WF_DIR = "/content/ComfyUI/user/default/workflows"

class ComfyClient:
    def __init__(self, base_url):
        self.base_url = base_url

    def queue_prompt(self, workflow):
        resp = requests.post(f"{self.base_url}/api/prompt", json={"prompt": workflow})
        resp.raise_for_status()
        return resp.json()["prompt_id"]

    def get_history(self, prompt_id):
        resp = requests.get(f"{self.base_url}/api/history/{prompt_id}")
        resp.raise_for_status()
        data = resp.json()
        return data.get(prompt_id)

    def wait_for_result(self, prompt_id, timeout=600, poll_interval=5):
        start = time.time()
        while time.time() - start < timeout:
            history = self.get_history(prompt_id)
            if history and history.get("status", {}).get("completed", False):
                return history
            if history and history.get("status", {}).get("status_str") == "error":
                raise RuntimeError("Генерация в ComfyUI завершилась с ошибкой")
            time.sleep(poll_interval)
        raise TimeoutError(f"Генерация не завершилась за {timeout} сек")

    def get_output_files(self, history):
        files = []
        for node_id, output in history.get("outputs", {}).items():
            for key in ["images", "gifs", "videos"]:
                for item in output.get(key, []):
                    fname = item.get("filename", "")
                    subfolder = item.get("subfolder", "")
                    fpath = os.path.join(OUTPUT_DIR, subfolder, fname) if subfolder else os.path.join(OUTPUT_DIR, fname)
                    if os.path.exists(fpath):
                        files.append(fpath)
        return files

comfy = ComfyClient(COMFY_URL)

# ── Загрузчики воркфлоу ─────────────────────────────────────────────

def load_workflow(name):
    path = os.path.join(WF_DIR, name)
    with open(path) as f:
        data = json.load(f)
    # Конвертация из сохранённого формата в API формат
    api_prompt = {}
    for node in data.get("nodes", []):
        node_id = str(node["id"])
        api_prompt[node_id] = {
            "class_type": node["type"],
            "inputs": {}
        }
        # Маппинг значений виджетов
        if "widgets_values" in node:
            api_prompt[node_id]["_widgets_values"] = node["widgets_values"]
        # Маппинг связанных входов
        if "inputs" in node:
            for inp in node["inputs"]:
                if inp.get("link") is not None:
                    # Поиск источника из массива links
                    for link in data.get("links", []):
                        if link[0] == inp["link"]:
                            src_node = str(link[1])
                            src_slot = link[2]
                            api_prompt[node_id]["inputs"][inp["name"]] = [src_node, src_slot]
    return api_prompt

def set_prompt_text(workflow_data, prompt_text, negative_text=None):
    """Находит ноды текстового кодирования и устанавливает текст промпта."""
    for node_id, node in workflow_data.items():
        ct = node.get("class_type", "")
        if ct == "WanVideoTextEncode":
            wv = node.get("_widgets_values", [])
            if len(wv) >= 1:
                wv[0] = prompt_text
            if negative_text and len(wv) >= 2:
                wv[1] = negative_text
    return workflow_data

def set_image_input(workflow_data, image_filename):
    """Находит ноды LoadImage и устанавливает имя файла."""
    for node_id, node in workflow_data.items():
        if node.get("class_type") in ("LoadImage", "LoadImageFromPath"):
            wv = node.get("_widgets_values", [])
            if len(wv) >= 1:
                wv[0] = image_filename
    return workflow_data

# ── Состояние ────────────────────────────────────────────────────────

current_job = {"active": False, "user_id": None, "prompt_id": None}

# ── Обработчики бота ─────────────────────────────────────────────────

async def start(update: Update, context: ContextTypes.DEFAULT_TYPE):
    await update.message.reply_text(
        "ComfyUI Видео Бот\n\n"
        "Команды:\n"
        "/photo <промпт> — ответьте на фото, чтобы анимировать его (клип 3.4с)\n"
        "/text <промпт> — генерация видео из текста\n"
        "/talking — ответьте на фото, прикрепите аудио\n"
        "/v2v <промпт> — ответьте на видео для рестайла\n"
        "/status — проверить текущую задачу\n"
        "/cancel — отменить текущую задачу\n\n"
        "Одна задача за раз. Генерация занимает 2-10 мин."
    )

async def status(update: Update, context: ContextTypes.DEFAULT_TYPE):
    if current_job["active"]:
        await update.message.reply_text(f"Задача выполняется (prompt_id: {current_job['prompt_id'][:8]}...)")
    else:
        await update.message.reply_text("Нет активных задач. Отправьте команду для начала!")

async def cancel(update: Update, context: ContextTypes.DEFAULT_TYPE):
    if current_job["active"]:
        try:
            requests.post(f"{COMFY_URL}/api/interrupt")
            current_job["active"] = False
            await update.message.reply_text("Задача отменена.")
        except Exception as e:
            await update.message.reply_text(f"Ошибка отмены: {e}")
    else:
        await update.message.reply_text("Нет активной задачи для отмены.")

async def generate_video(update: Update, context: ContextTypes.DEFAULT_TYPE, workflow_name, prompt_text, image_path=None, audio_path=None, video_path=None):
    if current_job["active"]:
        await update.message.reply_text("Другая задача уже выполняется. Используйте /status или /cancel.")
        return

    current_job["active"] = True
    current_job["user_id"] = update.effective_user.id

    try:
        status_msg = await update.message.reply_text(f"Запуск генерации с {workflow_name}...")

        # Загрузка и настройка воркфлоу
        # Для простоты используем эндпоинт ComfyUI API /api/prompt
        # JSON воркфлоу должен быть в API формате
        wf_path = os.path.join(WF_DIR, workflow_name)
        with open(wf_path) as f:
            workflow = json.load(f)

        # Отправка в очередь ComfyUI — используем файл воркфлоу напрямую
        # Примечание: /api/prompt ожидает API формат, а не сохранённый формат.
        # Используем более простой подход: сохраняем входные данные и используем эндпоинт очереди.

        await status_msg.edit_text("В очереди! Генерация... (2-10 мин)\nИспользуйте /status для проверки.")

        # Отслеживание новых файлов в папке вывода
        existing_files = set(glob.glob(f"{OUTPUT_DIR}/*.*"))
        start_time = time.time()
        timeout = 600  # 10 мин

        while time.time() - start_time < timeout:
            await asyncio.sleep(10)
            current_files = set(glob.glob(f"{OUTPUT_DIR}/*.*"))
            new_files = current_files - existing_files
            new_media = [f for f in new_files if f.endswith(('.mp4', '.png', '.gif'))]
            if new_media:
                for fpath in sorted(new_media):
                    if fpath.endswith('.mp4'):
                        await update.message.reply_video(video=open(fpath, 'rb'),
                            caption=f"Сгенерировано с {workflow_name}")
                    else:
                        await update.message.reply_photo(photo=open(fpath, 'rb'),
                            caption=f"Сгенерировано с {workflow_name}")
                await status_msg.edit_text("Готово!")
                break
        else:
            await status_msg.edit_text("Генерация превысила таймаут (10 мин). Проверьте интерфейс ComfyUI.")

    except Exception as e:
        await update.message.reply_text(f"Ошибка: {e}")
    finally:
        current_job["active"] = False
        current_job["prompt_id"] = None

async def cmd_text(update: Update, context: ContextTypes.DEFAULT_TYPE):
    prompt = " ".join(context.args) if context.args else None
    if not prompt:
        await update.message.reply_text("Использование: /text <ваш промпт>")
        return
    await update.message.reply_text(f"Промпт: {prompt}\nОткройте ComfyUI, загрузите воркфлоу video_wan_t2v, вставьте промпт и нажмите Queue.\nБот отслеживает новые файлы в папке вывода...")
    await generate_video(update, context, "video_wan_t2v.json", prompt)

async def cmd_photo(update: Update, context: ContextTypes.DEFAULT_TYPE):
    prompt = " ".join(context.args) if context.args else "animate this image, smooth motion, cinematic"
    reply = update.message.reply_to_message
    photo_file = None

    if reply and reply.photo:
        photo_file = await reply.photo[-1].get_file()
    elif update.message.photo:
        photo_file = await update.message.photo[-1].get_file()

    if not photo_file:
        await update.message.reply_text("Ответьте на фото командой /photo <промпт> или отправьте фото вместе с командой.")
        return

    # Скачивание фото
    fname = f"bot_input_{int(time.time())}.jpg"
    fpath = os.path.join(INPUT_DIR, fname)
    await photo_file.download_to_drive(fpath)
    await update.message.reply_text(f"Фото сохранено. Откройте ComfyUI, загрузите video_wan_clip, выберите {fname} как входное изображение, вставьте промпт и нажмите Queue.\nБот отслеживает вывод...")
    await generate_video(update, context, "video_wan_clip.json", prompt, image_path=fpath)

async def cmd_talking(update: Update, context: ContextTypes.DEFAULT_TYPE):
    reply = update.message.reply_to_message
    photo_file = None
    audio_file = None

    # Получение фото из ответа
    if reply and reply.photo:
        photo_file = await reply.photo[-1].get_file()

    # Получение аудио из текущего сообщения или ответа
    if update.message.audio:
        audio_file = await update.message.audio.get_file()
    elif update.message.voice:
        audio_file = await update.message.voice.get_file()
    elif reply and reply.audio:
        audio_file = await reply.audio.get_file()
    elif reply and reply.voice:
        audio_file = await reply.voice.get_file()

    if not photo_file or not audio_file:
        await update.message.reply_text(
            "Для говорящей головы:\n"
            "1. Отправьте портретное фото\n"
            "2. Ответьте на него командой /talking и прикрепите аудиофайл/голосовое сообщение"
        )
        return

    ts = int(time.time())
    photo_path = os.path.join(INPUT_DIR, f"bot_photo_{ts}.jpg")
    audio_path = os.path.join(INPUT_DIR, f"bot_audio_{ts}.wav")
    await photo_file.download_to_drive(photo_path)
    await audio_file.download_to_drive(audio_path)

    await update.message.reply_text(f"Фото + аудио сохранены. Загрузите video_wan_talking в ComfyUI, выберите входные файлы и нажмите Queue.\nБот отслеживает вывод...")
    await generate_video(update, context, "video_wan_talking.json", "talking head", image_path=photo_path, audio_path=audio_path)

async def cmd_v2v(update: Update, context: ContextTypes.DEFAULT_TYPE):
    prompt = " ".join(context.args) if context.args else None
    if not prompt:
        await update.message.reply_text("Использование: ответьте на видео командой /v2v <промпт>")
        return

    reply = update.message.reply_to_message
    video_file = None
    if reply and reply.video:
        video_file = await reply.video.get_file()
    elif reply and reply.document and reply.document.mime_type and reply.document.mime_type.startswith("video"):
        video_file = await reply.document.get_file()

    if not video_file:
        await update.message.reply_text("Ответьте на видео командой /v2v <промпт>")
        return

    ts = int(time.time())
    video_path = os.path.join(INPUT_DIR, f"bot_video_{ts}.mp4")
    await video_file.download_to_drive(video_path)

    await update.message.reply_text(f"Видео сохранено. Загрузите video_wan_v2v в ComfyUI, выберите {os.path.basename(video_path)} как входное видео, вставьте промпт и нажмите Queue.\nБот отслеживает вывод...")
    await generate_video(update, context, "video_wan_v2v.json", prompt, video_path=video_path)

# ── Запуск ───────────────────────────────────────────────────────────

if not BOT_TOKEN:
    print("ОШИБКА: Установите BOT_TOKEN выше!")
else:
    # Запуск ComfyUI
    print("=" * 60)
    print("ЭТАП 1/3: Запуск ComfyUI...")
    print("=" * 60)
    import os as _os
    _os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

    comfy_proc = subprocess.Popen(
        ["python", "main.py", "--listen", "0.0.0.0", "--port", "8188",
         "--enable-cors-header", "*"],
        cwd="/content/ComfyUI",
        stdout=open("/content/comfyui.log", "w"),
        stderr=subprocess.STDOUT
    )

    print("  Ожидание запуска ComfyUI...")
    for _i in range(24):
        time.sleep(5)
        try:
            r = requests.get(f"{COMFY_URL}/api/system_stats", timeout=3)
            if r.status_code == 200:
                gpu_name = r.json()['devices'][0]['name']
                print(f"  ComfyUI запущен за {(_i+1)*5} сек! GPU: {gpu_name}")
                break
        except Exception:
            pass
    else:
        print("  ComfyUI не ответил за 120 сек — проверьте /content/comfyui.log")
        with open("/content/comfyui.log") as _f:
            _log = _f.read()
        _errs = [l for l in _log.splitlines() if "error" in l.lower() or "import" in l.lower()]
        if _errs:
            print("  Возможные ошибки в логе:")
            for _e in _errs[:10]:
                print(f"    {_e}")

    # Установка cloudflared для доступа к интерфейсу
    print("\n" + "=" * 60)
    print("ЭТАП 2/3: Открытие туннеля Cloudflare...")
    print("=" * 60)
    subprocess.run(
        ["bash", "-c",
         "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1"],
        check=False
    )
    tunnel = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://localhost:8188"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    time.sleep(5)
    for _ in range(20):
        line = tunnel.stdout.readline().decode()
        match = re.search(r"https://[\w-]+\.trycloudflare\.com", line)
        if match:
            print(f"  Веб-интерфейс ComfyUI: {match.group(0)}")
            print("  (используйте для ручной загрузки воркфлоу и отладки)")
            print("  Если страница белая — перезапустите ячейку 5")
            break

    print("\n" + "=" * 60)
    print("ЭТАП 3/3: Запуск Telegram бота...")
    print("=" * 60)
    print()
    print("  Бот отслеживает папку вывода ComfyUI на наличие новых файлов.")
    print("  Для генерации используйте команды бота или веб-интерфейс ComfyUI.")
    print("  Бот автоматически отправит результаты в Telegram.")
    print()
    print("-" * 60)
    print("  ГОТОВО! Откройте бота в Telegram и отправьте /start")
    print("  Для остановки нажмите кнопку СТОП в Colab")
    print("-" * 60)
    print()

    app = Application.builder().token(BOT_TOKEN).build()
    app.add_handler(CommandHandler("start", start))
    app.add_handler(CommandHandler("text", cmd_text))
    app.add_handler(CommandHandler("photo", cmd_photo))
    app.add_handler(CommandHandler("talking", cmd_talking))
    app.add_handler(CommandHandler("v2v", cmd_v2v))
    app.add_handler(CommandHandler("status", status))
    app.add_handler(CommandHandler("cancel", cancel))

    # Запуск с polling (самый простой вариант для Colab)
    app.run_polling(allowed_updates=Update.ALL_TYPES)

---

## Руководство по использованию

### Подготовка (до запуска ноутбука)

| Шаг | Действие | Время |
|:---:|:---------|:-----:|
| 1 | Откройте [@BotFather](https://t.me/BotFather) в Telegram | 10 сек |
| 2 | Отправьте `/newbot`, придумайте имя и username | 30 сек |
| 3 | Скопируйте токен (понадобится в ячейке 5) | 10 сек |

---

### Как это работает

```
  ВЫ (Telegram)                    Google Colab
  ┌───────────┐              ┌──────────────────────┐
  │           │  /photo кот  │   ┌───────────────┐   │
  │ Telegram  │ -----------> │   │  Telegram бот  │   │
  │  клиент   │              │   └──────┬────────┘   │
  │           │              │          |             │
  │           │              │          v             │
  │           │              │   ┌───────────────┐   │
  │           │              │   │   ComfyUI API  │   │
  │           │              │   │  (генерация    │   │
  │           │              │   │   2-10 мин)    │   │
  │           │              │   └──────┬────────┘   │
  │           │   .mp4       │          |             │
  │           │ <----------- │   Готовое видео        │
  └───────────┘              └──────────────────────┘
```

**Пошагово:**
1. Вы отправляете команду боту в Telegram (например, `/photo <промпт>`)
2. Бот сохраняет ваши медиафайлы в папку ввода ComfyUI
3. Бот сообщает, какой воркфлоу загрузить и что настроить
4. Вы (или бот) запускаете генерацию в ComfyUI
5. Бот отслеживает папку вывода на наличие новых файлов
6. Результат автоматически отправляется вам в Telegram

---

### Примеры использования

**Анимация фото (самый популярный):**
1. Отправьте фото боту
2. Ответьте на него: `/photo девушка поворачивает голову и улыбается, cinematic`
3. Ждите 2-5 мин

**Видео из текста:**
```
/text кот в космосе на ракете, stars background, epic cinematic
```

**Говорящая голова:**
1. Отправьте портретное фото
2. Ответьте на него: `/talking` + прикрепите голосовое или аудиофайл

**Рестайл видео:**
1. Отправьте видео
2. Ответьте на него: `/v2v anime style, studio ghibli, detailed`

---

### Текущие ограничения

| Ограничение | Описание | Обходной путь |
|:------------|:---------|:-------------|
| Полуавтоматический режим | Нужно вручную загружать воркфлоу в ComfyUI | Экспортируйте воркфлоу в API формат (см. ниже) |
| Одна задача за раз | T4 не хватает VRAM для параллельной генерации | Используйте `/status` и `/cancel` |
| Таймаут сессии | Colab отключается через 90 мин простоя | Держите вкладку открытой |
| Размер файла | Telegram лимит — 50 МБ | Уменьшите разрешение или длительность |

### Полностью автоматический режим (продвинутый)

Чтобы бот работал без ручного взаимодействия с ComfyUI:
1. Откройте воркфлоу в ComfyUI
2. Экспортируйте его: **Menu -> Save as API** (формат `_api.json`)
3. Тогда бот сможет отправлять задачи напрямую через `/api/prompt`

---

### Решение проблем

| Проблема | Что проверить | Решение |
|:---------|:-------------|:--------|
| Бот не отвечает | Вывод ячейки 5 | Перезапустите ячейку 5, проверьте токен |
| ComfyUI не запускается | `/content/comfyui.log` | Перезапустите ячейки 1-4, затем 5 |
| Генерация зависла | `/status` в боте | `/cancel`, затем попробуйте снова |
| Ошибка нехватки памяти (OOM) | Настройки воркфлоу | Уменьшите разрешение или кадры |
| Туннель не открылся | Вывод ячейки 5 | Перезапустите ячейку 5 |
| Токен не принимается | Формат токена | Должен быть вида `123456:ABCdef...` от @BotFather |